# 06 · Image-size check (Step 3.1, D-047)

**Runtime: L4 GPU** (Runtime → Change runtime type → **L4 GPU**).

Scores the **untrained** model on the **validation set** (not test) at three image sizes, 0.26 / 0.5 / 1 megapixel, for answering (all 1,464 answerable val questions) and answerability (all 2,000 val questions). Then compares the sizes question by question (paired bootstrap).

The decision rule was written down before this run (DECISIONS.md D-047): use the smallest size within 1.0 point of the best answering accuracy and within 1.5 points of the best answerability AP.

Expect about **30–40 minutes**. A disconnect is safe: rerun all, and finished sizes are skipped.

Every command runs through `sh()`, which stops with an error if the command fails.

## 1 · Settings, Drive, code

In [ ]:
import os
import subprocess

from google.colab import drive

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
LOCAL = "/content/data"
RUNS = f"{DRIVE_ROOT}/runs"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"
RES = f"{RUNS}/resolution"
BUDGETS = [262144, 524288, 1048576]


def sh(cmd, **env):
    """Run a shell command, stream its output, and raise if it fails."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(
            f"Command failed (exit {proc.returncode}). Stop here and paste the output above."
        )


drive.mount("/content/drive")
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

**Paste back:** the commit line and the GPU line (should say NVIDIA L4).

## 2 · Pinned environment (~3–5 min)

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e {REPO_DIR} -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")

## 3 · Check label files, unzip photos (~3 min)
Our val split is carved from the official train split, so `train.zip` is needed.

In [ ]:
import hashlib
import json

expected = json.load(open(f"{REPO_DIR}/results/splits/files.json"))
for name in ("vqa_val.jsonl", "quality_val.jsonl"):
    digest = hashlib.sha256(open(f"{DATA}/processed/{name}", "rb").read()).hexdigest()
    assert digest == expected[name]["sha256"], f"{name} differs from the committed hash"
    print(f"{name}: matches committed hash")
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits train")

## 4 · Evaluate at each image size (~25–35 min)
One vLLM start per size. Sizes that already finished (both tasks in `metrics.json`) are skipped.

In [ ]:
%%time
for b in BUDGETS:
    out = f"{RES}/res-{b}"
    done = os.path.exists(f"{out}/metrics.json") and {"answer", "answerability"} <= set(
        json.load(open(f"{out}/metrics.json"))["tasks"]
    )
    if done:
        print(f"max_pixels={b}: already done, skipped")
        continue
    sh(
        f"{PY} -m vizwiz.eval.run_eval --records {DATA}/processed --images {LOCAL}/images --split val --tasks answer answerability --max-pixels {b} --run-id res-{b} --out {out}",
        HF_HOME="/content/hf_cache",
    )

**Paste back:** the six result lines that start with `[answer] vqa_accuracy` or `[answerability] average_precision`.

## 5 · Paired comparison and summary (~2 min)

In [ ]:
runs = " ".join(f"{RES}/res-{b}" for b in BUDGETS)
names = " ".join(str(b) for b in BUDGETS)
for task in ("answer", "answerability"):
    sh(
        f"{PY} -m vizwiz.eval.compare --records {DATA}/processed --split val --task {task} --runs {runs} --names {names} --reference 524288 --report {RES}/compare_{task}.json > /dev/null"
    )
summary = {
    "compare": {t: json.load(open(f"{RES}/compare_{t}.json")) for t in ("answer", "answerability")},
    "timing": {},
}
for b in BUDGETS:
    m = json.load(open(f"{RES}/res-{b}/metrics.json"))
    summary["timing"][str(b)] = {t: m["tasks"][t]["timing"] for t in ("answer", "answerability")}
json.dump(summary, open(f"{RES}/resolution_summary.json", "w"), indent=2)
for t, c in summary["compare"].items():
    print(t, c["metric"], {name: (r["value"], r["diff_ci95"]) for name, r in c["runs"].items()})

**Paste back:** the two lines printed (`answer vqa_accuracy {...}` and `answerability average_precision {...}`).

## 6 · Flush Drive, then disconnect
Upload back from `MyDrive/vizwiz/runs/resolution/`: **`resolution_summary.json`** only (aggregate numbers; prediction files stay on Drive).

Then Runtime → Disconnect and delete runtime.

### If Colab disconnects
Reconnect (L4) and Run all again: finished image sizes are skipped.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")